# Implementasi Uninformed Search: BFS dan DFS

## 1. Tujuan

Setelah mengerjakan notebook ini, kita diharapkan dapat:

- memahami konsep **state-space search**,
- memahami cara kerja **Breadth-First Search (BFS)** dan **Depth-First Search (DFS)**,
- mengimplementasikan BFS dan DFS **dari nol** (tanpa library search),
- menyelesaikan **Sudoku 4×4** menggunakan kedua algoritma,
- melihat **proses pencarian secara bertahap**.

## 2. Konsep Uninformed Search

**Uninformed search** (blind search) adalah kelompok algoritma pencarian yang **tidak memakai informasi tambahan** tentang seberapa dekat suatu state dengan tujuan (tidak ada heuristic). Algoritma hanya tahu cara membangkitkan state baru dan cara mengecek apakah sebuah state adalah tujuan.

| Istilah | Penjelasan | Pada Sudoku 4×4 |
|---|---|---|
| **State** | Gambaran kondisi masalah pada satu saat | Satu papan 4×4 dengan sebagian sel terisi |
| **Initial state** | State awal | Papan puzzle awal |
| **Goal state** | State yang dicari | Papan terisi penuh dan valid |
| **Successor** | State hasil satu aksi dari state saat ini | Papan baru setelah satu angka diletakkan |
| **Search tree** | Pohon: akar = initial state, anak = successor | Pohon papan-papan hasil pengisian sel |

**Perbedaan BFS dan DFS secara konseptual:**

- **BFS** menjelajah pohon **level demi level**: semua node kedalaman *d* diperiksa sebelum node kedalaman *d+1*.
- **DFS** menjelajah pohon **sedalam mungkin** lebih dulu pada satu cabang, lalu **backtrack** ke cabang lain bila buntu.

## 3. Breadth-First Search (BFS)

**Definisi.** BFS adalah algoritma pencarian yang mengeksplorasi node berdasarkan urutan kedalaman, dari yang paling dangkal.

**Cara kerja.**
1. Masukkan initial state ke dalam *queue*.
2. Ambil state paling depan dari queue.
3. Jika state itu goal, selesai.
4. Jika bukan, bangkitkan semua successor dan masukkan ke **belakang** queue.
5. Ulangi sampai solusi ditemukan atau queue kosong.

**Queue (FIFO).** *First In, First Out*: node yang masuk lebih dulu dieksplorasi lebih dulu. Karena successor selalu masuk di belakang, semua node level saat ini habis diproses sebelum node level berikutnya.

**Cara BFS memilih node.** Selalu mengambil node di **depan** queue, yaitu node yang paling lama menunggu (paling dangkal).

**Kelebihan:** menemukan solusi dengan langkah paling sedikit; lengkap (pasti menemukan solusi jika ada, pada ruang state berhingga).
**Kekurangan:** memori besar karena menyimpan seluruh node pada satu level (frontier lebar).
**Cocok digunakan** ketika solusi dangkal atau kita menginginkan jumlah langkah minimum.

**Pseudocode:**
```
BFS(initial_state):
    queue <- [initial_state]
    while queue tidak kosong:
        state <- ambil elemen DEPAN queue
        if state adalah goal:
            return state
        for setiap successor dari state:
            masukkan successor ke BELAKANG queue
    return gagal
```

## 4. Depth-First Search (DFS)

**Definisi.** DFS adalah algoritma pencarian yang mengeksplorasi satu cabang sedalam mungkin sebelum mencoba cabang lain.

**Cara kerja.**
1. Masukkan initial state ke dalam *stack*.
2. Ambil state paling atas dari stack.
3. Jika state itu goal, selesai.
4. Jika bukan, bangkitkan successor dan masukkan ke **atas** stack.
5. Jika buntu (tidak ada successor), otomatis *backtrack* karena elemen berikutnya di stack adalah cabang sebelumnya.

**Stack (LIFO).** *Last In, First Out*: node yang terakhir masuk dieksplorasi lebih dulu. DFS juga bisa ditulis dengan rekursi, karena *call stack* berperan sebagai stack. Di notebook ini kita memakai **stack eksplisit**.

**Cara DFS memilih node.** Selalu mengambil node di **atas** stack, yaitu node yang paling baru dibangkitkan (paling dalam).

**Kelebihan:** memori kecil (hanya menyimpan satu jalur beserta saudara-saudaranya); cepat jika solusi berada dalam.
**Kekurangan:** tidak menjamin solusi terdekat; bisa terjebak lama di cabang yang salah (dan tak berhingga pada ruang state tak terbatas).
**Cocok digunakan** ketika solusi berada pada kedalaman tertentu yang seragam (seperti Sudoku, semua solusi berada pada kedalaman = jumlah sel kosong) dan memori terbatas.

**Pseudocode:**
```
DFS(initial_state):
    stack <- [initial_state]
    while stack tidak kosong:
        state <- ambil elemen ATAS stack
        if state adalah goal:
            return state
        for setiap successor dari state:
            masukkan successor ke ATAS stack
    return gagal
```

## 5. Case Study: Sudoku 4×4

Sudoku 4×4 memakai angka **1–4**. Aturan: setiap **baris**, **kolom**, dan **blok 2×2** harus berisi angka 1–4 tanpa pengulangan.

Pemodelan sebagai masalah pencarian:

- **State**: sebuah papan Sudoku. Sel kosong ditulis **0**.
- **Action**: mengisi sebuah sel kosong dengan angka yang valid.
- **Successor**: papan baru setelah satu angka diletakkan.
- **Goal state**: papan yang seluruh selnya terisi dan valid.

> **Catatan desain.** Agar pohon pencarian tidak menggandakan state yang sama (mengisi sel A lalu B sama dengan B lalu A), setiap aksi selalu mengisi **sel kosong pertama** (urut baris, lalu kolom). Ini bukan heuristic dan bukan optimasi algoritma: hanya aturan untuk membangkitkan successor, sehingga BFS dan DFS memakai pohon pencarian yang sama.

In [1]:
# Puzzle Sudoku 4x4 (0 = sel kosong)
puzzle_awal = (
    (0, 0, 0, 0),
    (0, 0, 1, 2),
    (2, 0, 0, 3),
    (0, 3, 0, 0),
)

## 6. Representasi State

State disimpan sebagai **tuple of tuple** (immutable), sehingga state lama tidak berubah ketika kita membuat successor.

In [2]:
from collections import deque

UKURAN = 4
BLOK = 2

def tampilkan_board(board):
    """Menampilkan board sebagai grid; '.' untuk sel kosong."""
    for r in range(UKURAN):
        if r % BLOK == 0 and r != 0:
            print("------+-------")
        baris = ""
        for c in range(UKURAN):
            if c % BLOK == 0 and c != 0:
                baris += "| "
            baris += (str(board[r][c]) if board[r][c] != 0 else ".") + " "
        print(baris)

print("Initial state:")
tampilkan_board(puzzle_awal)

Initial state:
. . | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 


### Fungsi-fungsi pendukung

- `cari_sel_kosong(board)`: mengembalikan posisi `(baris, kolom)` sel kosong pertama, atau `None` jika papan penuh.
- `angka_valid(board, r, c, angka)`: mengecek apakah `angka` boleh diletakkan di `(r, c)` (tidak bentrok pada baris, kolom, dan blok 2×2).
- `bangkitkan_successor(board)`: menghasilkan daftar `(board_baru, (r, c, angka))` untuk sel kosong pertama.
- `is_goal(board)`: mengecek apakah papan sudah penuh. Karena angka hanya diletakkan jika valid, papan penuh pasti valid.

In [3]:
def cari_sel_kosong(board):
    for r in range(UKURAN):
        for c in range(UKURAN):
            if board[r][c] == 0:
                return (r, c)
    return None

def angka_valid(board, r, c, angka):
    # cek baris
    if angka in board[r]:
        return False
    # cek kolom
    for i in range(UKURAN):
        if board[i][c] == angka:
            return False
    # cek blok 2x2
    br, bc = (r // BLOK) * BLOK, (c // BLOK) * BLOK
    for i in range(br, br + BLOK):
        for j in range(bc, bc + BLOK):
            if board[i][j] == angka:
                return False
    return True

def bangkitkan_successor(board):
    """Mengembalikan list (board_baru, (r, c, angka))."""
    posisi = cari_sel_kosong(board)
    if posisi is None:
        return []
    r, c = posisi
    hasil = []
    for angka in range(1, UKURAN + 1):
        if angka_valid(board, r, c, angka):
            papan_baru = [list(baris) for baris in board]
            papan_baru[r][c] = angka
            papan_baru = tuple(tuple(baris) for baris in papan_baru)
            hasil.append((papan_baru, (r, c, angka)))
    return hasil

def is_goal(board):
    return cari_sel_kosong(board) is None

# Uji cepat
print("Successor dari initial state:")
for papan, (r, c, a) in bangkitkan_successor(puzzle_awal):
    print(f"  isi sel ({r},{c}) dengan {a}")

Successor dari initial state:
  isi sel (0,0) dengan 1
  isi sel (0,0) dengan 3
  isi sel (0,0) dengan 4


## 7. Implementasi BFS

BFS memakai `deque` dari `collections` sebagai **queue**: `append` untuk menaruh di belakang, `popleft` untuk mengambil dari depan. Setiap elemen queue berisi `(board, depth)`.

Parameter `max_progress` mengatur berapa langkah pertama yang dicetak (agar output tetap readable). Penghitung `jumlah_dieksplorasi` bertambah setiap kali sebuah state diambil dari queue.

In [4]:
def bfs(initial, max_progress=0):
    queue = deque()
    queue.append((initial, 0))
    jumlah_dieksplorasi = 0

    while queue:
        board, depth = queue.popleft()
        jumlah_dieksplorasi += 1

        tampil = jumlah_dieksplorasi <= max_progress
        if tampil:
            print(f"Langkah {jumlah_dieksplorasi} | depth={depth} | ukuran queue={len(queue)}")
            tampilkan_board(board)

        if is_goal(board):
            if tampil or max_progress > 0:
                print(f">>> SOLUSI DITEMUKAN pada langkah {jumlah_dieksplorasi} (depth={depth})")
            return board, jumlah_dieksplorasi

        successors = bangkitkan_successor(board)
        if tampil:
            r, c = cari_sel_kosong(board)
            if successors:
                angka_valid_list = [a for _, (_, _, a) in successors]
                print(f"Sel yang diisi: ({r},{c}) | angka valid: {angka_valid_list}")
                print(f"-> {len(successors)} successor masuk ke BELAKANG queue")
            else:
                print(f"Sel yang diisi: ({r},{c}) | tidak ada angka valid -> buntu (tidak ada successor)")
            print()

        for papan_baru, aksi in successors:
            queue.append((papan_baru, depth + 1))

    return None, jumlah_dieksplorasi

## 8. Progress BFS

Pada fungsi `bfs` di atas, progress dicetak untuk `max_progress` langkah pertama. Untuk tiap langkah ditampilkan: nomor langkah, kedalaman, state yang dieksplorasi, sel/angka yang dicoba, apakah ada successor, serta pesan ketika solusi ditemukan.

## 9. Jalankan BFS

In [5]:
print("=== INITIAL BOARD ===")
tampilkan_board(puzzle_awal)
print()
print("=== PROGRESS BFS (8 langkah pertama) ===\n")

solusi_bfs, jumlah_bfs = bfs(puzzle_awal, max_progress=8)

=== INITIAL BOARD ===
. . | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 

=== PROGRESS BFS (8 langkah pertama) ===

Langkah 1 | depth=0 | ukuran queue=0
. . | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 
Sel yang diisi: (0,0) | angka valid: [1, 3, 4]
-> 3 successor masuk ke BELAKANG queue

Langkah 2 | depth=1 | ukuran queue=2
1 . | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 
Sel yang diisi: (0,1) | angka valid: [2, 4]
-> 2 successor masuk ke BELAKANG queue

Langkah 3 | depth=1 | ukuran queue=3
3 . | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 
Sel yang diisi: (0,1) | angka valid: [1, 2, 4]
-> 3 successor masuk ke BELAKANG queue

Langkah 4 | depth=1 | ukuran queue=5
4 . | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 
Sel yang diisi: (0,1) | angka valid: [1, 2]
-> 2 successor masuk ke BELAKANG queue

Langkah 5 | depth=2 | ukuran queue=6
1 2 | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 
Sel yang diisi: (0,2) | angka valid: [3, 4]
-> 2 succe

In [6]:
# Jalankan ulang tanpa progress untuk mendapat hasil akhir
solusi_bfs, jumlah_bfs = bfs(puzzle_awal, max_progress=0)

print("=== SOLUTION BOARD (BFS) ===")
tampilkan_board(solusi_bfs)
print(f"\nJumlah state yang dieksplorasi (BFS): {jumlah_bfs}")

=== SOLUTION BOARD (BFS) ===
1 2 | 3 4 
3 4 | 1 2 
------+-------
2 1 | 4 3 
4 3 | 2 1 

Jumlah state yang dieksplorasi (BFS): 27


## 10. Implementasi DFS

DFS memakai `list` Python sebagai **stack**: `append` untuk menaruh di atas, `pop` untuk mengambil dari atas. Successor dimasukkan dalam **urutan terbalik** agar angka terkecil berada di atas stack dan dieksplorasi lebih dulu (urutan ini hanya untuk menjaga keterbacaan).

Elemen stack berisi `(board, depth)`, sehingga perpindahan depth (maju/mundur) dapat dilihat.

In [7]:
def dfs(initial, max_progress=0):
    stack = []
    stack.append((initial, 0))
    jumlah_dieksplorasi = 0
    depth_sebelumnya = 0

    while stack:
        board, depth = stack.pop()
        jumlah_dieksplorasi += 1

        tampil = jumlah_dieksplorasi <= max_progress
        if tampil:
            if depth > depth_sebelumnya:
                arah = "turun lebih dalam"
            elif depth < depth_sebelumnya:
                arah = "BACKTRACK ke cabang lain"
            else:
                arah = "depth sama (saudara)"
            print(f"Langkah {jumlah_dieksplorasi} | depth {depth_sebelumnya} -> {depth} ({arah}) | ukuran stack={len(stack)}")
            tampilkan_board(board)
        depth_sebelumnya = depth

        if is_goal(board):
            if max_progress > 0:
                print(f">>> SOLUSI DITEMUKAN pada langkah {jumlah_dieksplorasi} (depth={depth})")
            return board, jumlah_dieksplorasi

        successors = bangkitkan_successor(board)
        if tampil:
            r, c = cari_sel_kosong(board)
            if successors:
                pilihan = [a for _, (_, _, a) in successors]
                print(f"Sel yang diisi: ({r},{c}) | pilihan successor: {pilihan}")
                print(f"-> angka {pilihan[0]} akan dieksplorasi berikutnya (paling atas stack)")
            else:
                print(f"Sel yang diisi: ({r},{c}) | tidak ada angka valid -> buntu, akan backtrack")
            print()

        for papan_baru, aksi in reversed(successors):
            stack.append((papan_baru, depth + 1))

    return None, jumlah_dieksplorasi

## 11. Jalankan DFS

In [11]:
print("=== INITIAL BOARD ===")
tampilkan_board(puzzle_awal)
print()
print("=== PROGRESS DFS (8 langkah pertama) ===\n")

solusi_dfs, jumlah_dfs = dfs(puzzle_awal, max_progress=8)

=== INITIAL BOARD ===
. . | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 

=== PROGRESS DFS (8 langkah pertama) ===

Langkah 1 | depth 0 -> 0 (depth sama (saudara)) | ukuran stack=0
. . | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 
Sel yang diisi: (0,0) | pilihan successor: [1, 3, 4]
-> angka 1 akan dieksplorasi berikutnya (paling atas stack)

Langkah 2 | depth 0 -> 1 (turun lebih dalam) | ukuran stack=2
1 . | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 
Sel yang diisi: (0,1) | pilihan successor: [2, 4]
-> angka 2 akan dieksplorasi berikutnya (paling atas stack)

Langkah 3 | depth 1 -> 2 (turun lebih dalam) | ukuran stack=3
1 2 | . . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 
Sel yang diisi: (0,2) | pilihan successor: [3, 4]
-> angka 3 akan dieksplorasi berikutnya (paling atas stack)

Langkah 4 | depth 2 -> 3 (turun lebih dalam) | ukuran stack=4
1 2 | 3 . 
. . | 1 2 
------+-------
2 . | . 3 
. 3 | . . 
Sel yang diisi: (0,3) | pilihan successor: [4]
-> ang

In [9]:
# Jalankan ulang tanpa progress untuk mendapat hasil akhir
solusi_dfs, jumlah_dfs = dfs(puzzle_awal, max_progress=0)

print("=== SOLUTION BOARD (DFS) ===")
tampilkan_board(solusi_dfs)
print(f"\nJumlah state yang dieksplorasi (DFS): {jumlah_dfs}")

=== SOLUTION BOARD (DFS) ===
1 2 | 3 4 
3 4 | 1 2 
------+-------
2 1 | 4 3 
4 3 | 2 1 

Jumlah state yang dieksplorasi (DFS): 12


## 13. Analisis

Ringkasan hasil eksperimen pada puzzle yang sama:

In [10]:
print(f"{'Algoritma':<10}{'State dieksplorasi':>22}")
print("-" * 32)
print(f"{'BFS':<10}{jumlah_bfs:>22}")
print(f"{'DFS':<10}{jumlah_dfs:>22}")

Algoritma     State dieksplorasi
--------------------------------
BFS                           27
DFS                           12

Solusi BFS sama dengan solusi DFS? True


## Kesimpulan

- Masalah Sudoku 4×4 dapat dimodelkan sebagai **state-space search**: state = papan, action = mengisi sel kosong dengan angka valid, successor = papan baru, goal = papan penuh yang valid.
- **BFS** memakai **queue (FIFO)**, mengeksplorasi state level demi level dan berhenti ketika state goal diambil dari queue.
- **DFS** memakai **stack (LIFO)**, menelusuri satu cabang sedalam mungkin dan melakukan **backtrack** ketika buntu.
- Keduanya memakai fungsi successor dan pengecekan goal yang sama; yang membedakan hanya cara frontier dikelola, dan itu menghasilkan urutan eksplorasi yang berbeda.